# AlphaGo in miniature: tree search on a $5\times5$ Go board

*others · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/alphago_mini)

One self-play game of Go on a tiny board, chosen by the same search AlphaGo used: Monte Carlo tree search with the PUCT rule. The Go rules themselves are tensor operations: captures and the area score are flood fills, one einsum with the board's adjacency matrix per step. Each new position is judged by a whole batch of random games played at once. Watch the visit counts concentrate on the move the search believes in.

**Vector form**

$$a^* = \arg\max_a \Big[Q(s,a) + c\,P(s,a)\,\frac{\sqrt{\sum_b N(s,b)}}{1 + N(s,a)}\Big],\qquad \pi(a\,|\,s) = \frac{N(s,a)}{\sum_b N(s,b)}$$

**Index (tensor) form**

$$r_{bp} \leftarrow r_{bp} \lor \big(s_{bp} \land [A_{pq}\,r_{bq} > 0]\big),\qquad U_a = Q_a + c\,P_a\,\frac{\sqrt{N_b \mathbf 1_b}}{1 + N_a},\qquad v = \frac1R\sum_{r=1}^{R}\operatorname{sign}\big(\text{score}_r - k\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $X_p$ | `empty_board(n)` | (n²,) | point $p$ of the board: $+1$ black, $-1$ white, 0 empty |
| $A_{pq}$ | `adjacency(n)` | (n², n²) | the board as a graph: 1 when $p$ and $q$ are horizontal or vertical neighbours |
| $r \leftarrow r \lor (s \land A r)$ | `reach(seed, allowed, A)  # einsum('pq,bq->bp', A, r)` | (B, n²) | flood fill: from the empty points through a colour finds the groups with a liberty; the rest is captured |
| $\text{score}_b$ | `score(board, A, komi)` | (B,) | area scoring: stones plus empty points only that colour can reach, black minus white minus komi $k$ |
| $v$ | `playouts(board, player, A, R, depth, komi)` | () | $R$ random games played as one $(R, n^2)$ tensor; mean result for black; never filling one's own eye |
| $U_a$ | `puct(q, prior, visits, c)` | (moves,) | PUCT: value $Q_a$ plus an exploration bonus that shrinks as move $a$ is visited; uniform prior $P_a$ |
| $\pi_a = N_a / \sum_b N_b$ | `search(root, A, sims, ...)` | (n²+1,) | visit counts after all simulations (last entry: pass); the most visited move is played |

Each simulation walks down the tree choosing the PUCT maximum, adds one new position, scores it with the batch of random playouts and adds that result to every move on the path. Good moves collect visits, so the search concentrates where it matters without looking at every line. AlphaGo used the same loop but replaced the uniform prior $P_a$ with a policy network and the random playouts with a value network, both trained on games; AlphaGo Zero learned both purely from self-play. On $5\times5$ the centre opening is strong for black, and with few simulations the search makes visible mistakes; raise the simulations and the game gets sharper but slower. Rules: Tromp–Taylor (suicide removes one's own group, area scoring) with a simple ko check.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/alphago_mini/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import score


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = params["board_size"]
    adj = tensors.adjacency(n, dev)  # (P, P)
    board = tensors.empty_board(n, dev)  # (P,)
    max_moves = 6 if fast_mode() else n * n + n
    print(
        f"{n}×{n} Go, komi {params['komi']}, {params['simulations']} simulations per move, "
        f"{params['rollouts']} random playouts per leaf"
    )

    boards, moves, counts, values = solve.self_play(
        board, adj, params["simulations"], params["c_puct"], params["rollouts"], n * n, params["komi"], max_moves, 0
    )
    for i, (m, v) in enumerate(zip(moves.tolist(), values.tolist())):
        colour = "black" if i % 2 == 0 else "white"
        where = "pass" if m == n * n else f"({m // n + 1}, {m % n + 1})"
        if i < 6 or i == len(moves) - 1:
            print(f"move {i + 1:2d}: {colour:5s} plays {where:8s}  search thinks black wins with p = {(v + 1) / 2:.2f}")
    final = score(boards[-1][None], adj, params["komi"]).item()
    print(f"final position after {len(moves)} moves:\n{solve.show(boards[-1], n)}")
    print(f"area score black − white − komi = {final:+.1f}  →  {'black' if final > 0 else 'white'} wins")
    return plot.animate(boards, moves, counts, values, n, out_dir / "alphago_mini.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/alphago_mini/equation.py
r"""Rules and search → tensor translation.

The board is a graph: points p, q are neighbours when A_{pq} = 1. Growing a set of points r by one step is

    (A r)_p = A_{pq} r_q > 0       (q summed),

and every Go rule here is a "flood fill" built from it: starting from a seed set, repeatedly add points of an
allowed set that touch what has been reached,

    r \leftarrow r \lor (s \land A r)      until nothing changes.

* A group is alive if it has a liberty: seed = empty points, allowed = stones of that colour. Unreached stones
  are captured (opponent first, then the mover's own stones: Tromp–Taylor rules).
* Area score: seed = black stones, allowed = empty points gives the territory black reaches; the same for
  white; an empty point counts for a colour only if the other colour cannot reach it.

AlphaGo's search (Monte Carlo tree search with PUCT) picks at each node the move

    a^* = \arg\max_a \Big[ Q_a + c\,P_a\,\frac{\sqrt{\sum_b N_b}}{1 + N_a} \Big],

where N_a counts visits, Q_a is the mean value of the move, and P_a is a prior. After all simulations it
plays in proportion to the visit counts, π_a = N_a / Σ_b N_b.
"""

import torch


def grow(r: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
    r"""(A r)_p > 0: points touching the set r.   r: (B, P) bool → (B, P) bool"""
    return torch.einsum("pq,bq->bp", adj, r.float()) > 0


def reach(seed: torch.Tensor, allowed: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
    r"""Flood fill r ← r ∨ (s ∧ A r) from seed through allowed points.   (B, P) bool → (B, P) bool"""
    r = seed
    while True:
        nxt = r | (allowed & grow(r, adj))
        if torch.equal(nxt, r):
            return r
        r = nxt


def play(board: torch.Tensor, player: torch.Tensor, move: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
    r"""Place a stone (move = P means pass), then capture groups without liberties.
    board: (B, P) in {−1, 0, 1}, player: (B,), move: (B,) → (B, P)"""
    b = board.clone()
    n_points = b.shape[1]
    stone = move < n_points
    b[stone, move[stone]] = player[stone]
    for colour in (-player, player):  # opponent's groups first, then suicide of one's own
        mine = b == colour[:, None]
        alive = reach(b == 0, mine, adj)
        b[mine & ~alive] = 0
    return b


def score(board: torch.Tensor, adj: torch.Tensor, komi: float) -> torch.Tensor:
    r"""Area score for black: stones + empty points only black reaches, minus the same for white, minus komi.
    (B, P) → (B,)"""
    empty = board == 0
    black_reach = reach(board > 0, empty, adj)
    white_reach = reach(board < 0, empty, adj)
    black = (board > 0) | (empty & black_reach & ~white_reach)
    white = (board < 0) | (empty & white_reach & ~black_reach)
    return black.sum(-1).float() - white.sum(-1).float() - komi


def puct(q: torch.Tensor, prior: torch.Tensor, visits: torch.Tensor, c: float) -> torch.Tensor:
    r"""U_a = Q_a + c P_a √(Σ_b N_b) / (1 + N_a): the score PUCT maximises.   (M,) → (M,)"""
    return q + c * prior * visits.sum().sqrt() / (1 + visits)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/alphago_mini/tensors.py
"""Build the concrete tensors: the board graph (adjacency matrix) and the empty board."""

import torch

BLACK, WHITE = 1.0, -1.0


def adjacency(n: int, device: torch.device) -> torch.Tensor:
    """A_{pq} = 1 if points p and q are horizontal or vertical neighbours on the n×n board: shape (n², n²)."""
    a = torch.zeros(n * n, n * n, device=device)
    for r in range(n):
        for c in range(n):
            p = r * n + c
            if c + 1 < n:
                a[p, p + 1] = a[p + 1, p] = 1
            if r + 1 < n:
                a[p, p + n] = a[p + n, p] = 1
    return a


def empty_board(n: int, device: torch.device) -> torch.Tensor:
    """X_p ∈ {+1 black, −1 white, 0 empty} for the n² points, flattened row by row: shape (n²,)."""
    return torch.zeros(n * n, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/alphago_mini/solve.py
"""AlphaGo-style search: a tree of positions explored with PUCT, leaves judged by a batch of random playouts."""

import math
from dataclasses import dataclass, field

import torch

from .equation import play, puct, score


def legal_moves(board: torch.Tensor, player: float, previous: torch.Tensor | None, adj: torch.Tensor):
    """Try all P points + pass at once (one batched play): legal if the point was empty, the new stone survives
    (no suicide) and the position does not repeat the one before the opponent's move (simple ko).
    Returns a mask (P+1,) and the resulting boards (P+1, P)."""
    n_points = board.shape[0]
    moves = torch.arange(n_points + 1, device=board.device)
    after = play(board.expand(n_points + 1, -1), torch.full((n_points + 1,), player, device=board.device), moves, adj)
    legal = torch.ones(n_points + 1, dtype=torch.bool, device=board.device)
    legal[:n_points] = (board == 0) & (after[torch.arange(n_points), torch.arange(n_points)] == player)
    if previous is not None:
        legal[:n_points] &= ~(after[:n_points] == previous).all(-1)
    return legal, after


def playouts(board, player: float, adj, n: int, depth: int, komi: float, gen: torch.Generator) -> float:
    """Play n random games at once from this position for `depth` moves; random moves never fill one's own eye.
    Returns the mean result for black: +1 black wins, −1 white wins."""
    b = board.expand(n, -1).clone()
    to_play = torch.full((n,), player, device=board.device)
    degree = adj.sum(1)  # neighbours of every point (2 at corners, 3 on edges, 4 inside)
    for _ in range(depth):
        own = torch.einsum("pq,bq->bp", adj, (b == to_play[:, None]).float())
        eye = (b == 0) & (own == degree)  # an empty point surrounded by one's own stones
        weights = torch.cat([((b == 0) & ~eye).float(), torch.full((n, 1), 1e-3, device=b.device)], 1)
        move = torch.multinomial(weights, 1, generator=gen)[:, 0]
        b = play(b, to_play, move, adj)
        to_play = -to_play
    return torch.sign(score(b, adj, komi)).mean().item()


@dataclass
class Node:
    board: torch.Tensor
    to_play: float
    previous: torch.Tensor | None  # board before the last move, for the ko rule
    passes: int = 0  # consecutive passes leading here; 2 ends the game
    moves: torch.Tensor | None = None  # legal move indices, filled on expansion
    children: list = field(default_factory=list)
    prior: torch.Tensor | None = None
    visits: torch.Tensor | None = None
    value_sum: torch.Tensor | None = None  # summed values for black


def expand(node: Node, adj) -> None:
    legal, after = legal_moves(node.board, node.to_play, node.previous, adj)
    node.moves = legal.nonzero()[:, 0]
    n_points = node.board.shape[0]
    p = torch.ones(len(node.moves), device=node.board.device)
    p[node.moves == n_points] = 0.1  # uniform prior; passing is discouraged (AlphaGo uses a policy network here)
    node.prior = p / p.sum()
    node.visits = torch.zeros(len(node.moves), device=node.board.device)
    node.value_sum = torch.zeros(len(node.moves), device=node.board.device)
    node.children = [
        Node(after[m], -node.to_play, node.board, node.passes + 1 if m == n_points else 0) for m in node.moves.tolist()
    ]


def search(root: Node, adj, sims: int, c: float, rollouts: int, depth: int, komi: float, gen) -> torch.Tensor:
    """Run `sims` simulations from root; return its visit counts N_a for all P+1 moves."""
    if root.moves is None:
        expand(root, adj)
    for _ in range(sims):
        node, path = root, []
        while node.moves is not None and node.passes < 2:  # selection: follow the PUCT maximum
            # mean value of each move from the mover's side; untried moves start at the node's average
            parent_mean = node.value_sum.sum() / node.visits.sum().clamp(min=1)
            q = torch.where(node.visits > 0, node.value_sum / node.visits.clamp(min=1), parent_mean) * node.to_play
            k = int(torch.argmax(puct(q, node.prior, node.visits, c)))
            path.append((node, k))
            node = node.children[k]
        if node.passes >= 2:  # both passed: the game is over, count the score exactly
            value = math.copysign(1.0, score(node.board[None], adj, komi).item())
        else:  # expansion + evaluation by random playouts
            expand(node, adj)
            value = playouts(node.board, node.to_play, adj, rollouts, depth, komi, gen)
        for parent, k in path:  # backup
            parent.visits[k] += 1
            parent.value_sum[k] += value
    counts = torch.zeros(root.board.shape[0] + 1, device=root.board.device)
    counts[root.moves] = root.visits
    return counts


def self_play(board, adj, sims: int, c: float, rollouts: int, depth: int, komi: float, max_moves: int, seed: int):
    """One game of the search against itself. Returns boards (M+1, P), moves (M,), visit counts (M, P+1) and
    the root value estimates for black (M,)."""
    gen = torch.Generator().manual_seed(seed)
    node = Node(board, 1.0, None)
    boards, moves, counts, values = [board], [], [], []
    while node.passes < 2 and len(moves) < max_moves:
        n = search(node, adj, sims, c, rollouts, depth, komi, gen)
        k = int(torch.argmax(node.visits))  # play the most visited move
        values.append((node.value_sum.sum() / node.visits.sum()).item())
        moves.append(int(node.moves[k]))
        counts.append(n)
        node = node.children[k]  # reuse the subtree for the next move
        boards.append(node.board)
    return torch.stack(boards), torch.tensor(moves), torch.stack(counts), torch.tensor(values)


def show(board: torch.Tensor, n: int) -> str:
    """Text picture of the board: X black, O white, . empty."""
    chars = {1.0: "X", -1.0: "O", 0.0: "."}
    return "\n".join(" ".join(chars[v] for v in board.view(n, n)[r].tolist()) for r in range(n))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/alphago_mini/plot.py
"""Replay the game: the board, the search's visit counts for each move, and its win estimate."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(boards: torch.Tensor, moves: torch.Tensor, counts: torch.Tensor, values: torch.Tensor, n: int, path: Path):
    """boards (M+1, P), moves (M,), counts (M, P+1), values (M,) for black."""
    bd, mv, ct = boards.cpu().numpy(), moves.cpu().numpy(), counts.cpu().numpy()
    win = (values.cpu().numpy() + 1) / 2
    n_moves = len(mv)

    fig, (ax_b, ax_v, ax_w) = plt.subplots(1, 3, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1, 1.2]})
    for ax in (ax_b, ax_v):
        ax.set_xlim(-0.6, n - 0.4)
        ax.set_ylim(n - 0.4, -0.6)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
    ax_b.set_facecolor("#e8c27a")
    for k in range(n):
        ax_b.plot([0, n - 1], [k, k], color="#5b4636", lw=1)
        ax_b.plot([k, k], [0, n - 1], color="#5b4636", lw=1)
    stones = ax_b.scatter([], [], s=1400 / n * 3, edgecolors="#111827", linewidths=1)
    last = ax_b.scatter([], [], s=60, color="#dc2626")
    title_b = ax_b.set_title("", fontsize=10)

    heat = ax_v.imshow(np.zeros((n, n)), cmap="Blues", vmin=0, vmax=1)
    title_v = ax_v.set_title("", fontsize=10)

    ax_w.plot(np.arange(1, n_moves + 1), win, color="#9ca3af", lw=1)
    (dot,) = ax_w.plot([], [], "o-", color="#2563eb", lw=2)
    ax_w.axhline(0.5, color="#374151", lw=0.8, ls="--")
    ax_w.set_xlim(0.5, max(n_moves, 2) + 0.5)
    ax_w.set_ylim(0, 1)
    ax_w.set_xlabel("move")
    ax_w.set_ylabel("P(black wins), search estimate")
    ax_w.set_title("value of the root position", fontsize=10)
    ax_w.grid(True, alpha=0.2)

    def update(i: int):
        b = bd[i + 1].reshape(n, n)
        r, c = np.nonzero(b)
        stones.set_offsets(np.c_[c, r] if len(r) else np.empty((0, 2)))
        stones.set_facecolors(["#111827" if b[y, x] > 0 else "#ffffff" for y, x in zip(r, c)])
        m = mv[i]
        last.set_offsets([[m % n, m // n]] if m < n * n else np.empty((0, 2)))
        who = "black" if i % 2 == 0 else "white"
        title_b.set_text(f"move {i + 1}: {who} {'passes' if m == n * n else 'plays'}")
        visits = ct[i][: n * n].reshape(n, n)
        heat.set_data(visits / max(visits.max(), 1))
        title_v.set_text(f"visit counts N_a ({int(ct[i].sum())} sims)")
        dot.set_data(np.arange(1, i + 2), win[: i + 1])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n_moves, n_moves), path, fps=2)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'board_size': 5,
    'simulations': 48,
    'rollouts': 32,
    'c_puct': 1.5,
    'komi': 2.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.alphago_mini")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.alphago_mini.main")
problem_dir = Path("problems/others/alphago_mini")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
